# Tugas 3 Retrieval Informasi — Pencarian Dokumen pada BEIR SciFact

**Pencarian leksikal (BM25) vs. pencarian semantik (*dense retrieval*, sentence-transformers)**

Belva Fahrozi Chiangmaitri — Pengambilan Informasi (Retrieval Informasi), S2 Magister Teknik Informatika UDINUS

Seluruh logika berada di folder `src/` repositori; *notebook* ini hanya memanggil fungsi-fungsinya per tahap.
Jalankan sel secara berurutan. Disarankan memakai *runtime* GPU (Runtime → Change runtime type → T4 GPU).

## Tahap 1 — Menyiapkan pustaka

Repositori di-*clone* ke `/content` (atau diperbarui dengan `git pull` bila sudah ada), lalu paket yang belum tersedia
di Colab dipasang dari `requirements-colab.txt`. Paket bawaan Colab (torch, numpy, pandas, scikit-learn) tidak diganti;
versi yang benar-benar terpakai dicatat di `results/environment.json`.

> **Catatan:** jika Colab meminta *restart runtime* setelah pemasangan paket, lakukan *restart* lalu jalankan ulang dari Tahap 1.

In [ ]:
import os
REPO_DIR = "/content/RetrievalInformasi_Tugas3_P31.2025.02702_BelvaFC"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/belvahector-ship-it/RetrievalInformasi_Tugas3_P31.2025.02702_BelvaFC.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import sys, torch
from importlib import metadata

print("Python:", sys.version.split()[0])
for pkg in ["torch", "sentence-transformers", "transformers", "rank-bm25",
            "scikit-learn", "pandas", "numpy", "tqdm", "ir-measures"]:
    try:
        print(f"{pkg:22s}", metadata.version(pkg))
    except metadata.PackageNotFoundError:
        print(f"{pkg:22s}", "TIDAK TERPASANG")
print("GPU tersedia:", torch.cuda.is_available(),
      "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import pandas as pd
from IPython.display import display

from src import config
from src import run_pipeline as rp

# SMOKE = True hanya untuk uji cepat (korpus 200 dokumen); untuk laporan gunakan False.
SMOKE = False
OUT_DIR = rp.default_out_dir(SMOKE)
pd.set_option("display.max_colwidth", 120)
print("Seed:", config.SEED, "| k:", config.K, "| top-N untuk fusi:", config.TOP_N)

## Tahap 2 — Membaca file dataset (4 file)

File zip resmi BEIR SciFact diunduh, *checksum* sha256 diverifikasi, lalu empat file dibaca:
`corpus.jsonl`, `queries.jsonl`, `qrels/train.tsv`, dan `qrels/test.tsv`.

In [ ]:
data = rp.prepare_data(smoke=SMOKE)
display(rp.dataset_overview(data))

In [ ]:
for name, df in rp.file_examples(data, n=3).items():
    print(f"\nContoh isi {name}:")
    display(df)

## Tahap 3 — Mengambil 5 kueri secara acak

Kueri diambil dengan `random.Random(42).sample(sorted(test_query_ids), 5)`, hanya dari kueri yang memiliki *qrels* split *test*.

In [ ]:
rq = rp.random_queries_table(data)
display(rq)

## Tahap 4 — Pencarian BM25 (leksikal)

`rank_bm25.BM25Okapi` dengan parameter *default* pustaka. Praproses: *lowercase*, token `[a-z0-9]+`,
penghapusan *stop words* bahasa Inggris scikit-learn, tanpa *stemming*. Teks dokumen = judul + isi.

In [ ]:
runs, timings = {}, []
runs["BM25"], t = rp.run_bm25(data)
timings.append(t)
print("Parameter BM25:", {k: v for k, v in t.items() if k.startswith("bm25_")})

In [ ]:
for qid in data["random_qids"]:
    print(f"\nKueri {qid}: {data['queries'][qid]}")
    display(rp.top10_table(runs["BM25"], data, qid))

## Tahap 5 — Pencarian *sentence-transformers* (dense/semantik)

Dua model dengan revisi terkunci:
- `all-MiniLM-L6-v2` — *cosine similarity* (embedding dinormalisasi).
- `msmarco-distilbert-base-tas-b` — *dot product* tanpa normalisasi (sesuai *model card*).

In [ ]:
for key in config.DENSE_MODELS:
    print(f"\n=== {key}: {config.DENSE_MODELS[key]['name']} ===")
    runs[key], t = rp.run_dense(data, key)
    timings.append(t)

In [ ]:
for key in config.DENSE_MODELS:
    for qid in data["random_qids"]:
        print(f"\n[{key}] Kueri {qid}: {data['queries'][qid]}")
        display(rp.top10_table(runs[key], data, qid))

## Tahap 6 — Menghitung kinerja leksikal dan semantik

Metrik pada k = 10: nDCG@10, Precision@10, Recall@10, MRR@10, MAP@10 (relevansi biner dari *qrels*).
Dua cakupan: (a) 5 kueri acak sesuai instruksi, (b) seluruh 300 kueri *test* sebagai validasi.

In [ ]:
base = ["BM25", "MiniLM", "TAS-B"]
summary5, long5 = rp.evaluate_all({m: runs[m] for m in base}, data, data["random_qids"])
print("Cakupan (a): 5 kueri acak")
display(summary5.round(4))

In [ ]:
if SMOKE:
    print("Mode smoke: cakupan 300 kueri dilewati.")
else:
    summary300, long300 = rp.evaluate_all({m: runs[m] for m in base}, data, data["eval_qids"])
    print("Cakupan (b): seluruh kueri test")
    display(summary300.round(4))

## Tahap 7 — Analisis

Bahan analisis: perbandingan per kueri, *overlap* top-10 antara BM25 dan model *dense*, serta waktu komputasi.

*Analisis ditulis di laporan.*

In [ ]:
print("nDCG@10 per kueri (5 kueri acak):")
display(rp.per_query_comparison(long5, "nDCG@10").round(4))
print("Metrik lengkap per kueri:")
display(long5.round(4))

In [ ]:
print("Overlap top-10 BM25 vs model dense (5 kueri acak):")
display(rp.overlap_table(runs, data["random_qids"]).round(4))
if not SMOKE:
    ov = rp.overlap_table(runs, data["eval_qids"])
    print("Rata-rata overlap top-10 (seluruh kueri test):")
    display(ov.groupby("pair")[["overlap@10", "jaccard"]].mean().round(4))

In [ ]:
print("Waktu komputasi (detik):")
display(pd.DataFrame(timings).round(3))

## Bonus — *Hybrid retrieval* dengan Reciprocal Rank Fusion (RRF)

`score(d) = Σ 1 / (60 + rank)` atas daftar top-100 BM25 dan model *dense* (Cormack dkk., 2009).

In [ ]:
hyb, t = rp.run_hybrid(runs)
runs.update(hyb)
timings.extend(t)
for method in ["RRF-BM25+MiniLM", "RRF-BM25+TAS-B"]:
    for qid in data["random_qids"]:
        print(f"\n[{method}] Kueri {qid}: {data['queries'][qid]}")
        display(rp.top10_table(runs[method], data, qid))

In [ ]:
summary5_all, long5_all = rp.evaluate_all(runs, data, data["random_qids"])
print("Semua metode — 5 kueri acak")
display(summary5_all.round(4))
if not SMOKE:
    summary300_all, long300_all = rp.evaluate_all(runs, data, data["eval_qids"])
    print("Semua metode — seluruh kueri test")
    display(summary300_all.round(4))

### Menyimpan keluaran ke folder `results/`

Memakai fungsi yang sama dengan CLI (`rp.save_outputs`), termasuk validasi silang metrik dengan pustaka `ir-measures`:
perbandingan dengan skor berbasis peringkat (harus identik) dan dengan skor asli (selisih kecil mungkin muncul
karena aturan pemecahan skor seri yang berbeda).

In [ ]:
saved = rp.save_outputs(runs, timings, data, OUT_DIR)
if saved["crosscheck"] is not None:
    display(saved["crosscheck"])

In [ ]:
# Unduh seluruh folder results/ sebagai zip
import shutil
zip_path = shutil.make_archive(str(config.ROOT_DIR / "results_scifact"), "zip", config.RESULTS_DIR)
print("Arsip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Bukan lingkungan Colab: unduh berkas zip secara manual dari path di atas.")